# The Eye of Sauron: Ring Corruption Event Stream

This notebook streams synthetic "Ring corruption" telemetry into a Fabric Eventstream.
An Activator rule watches the corruption level and fires when the Ring is detected.

**The journey:** the Ringbearer travels toward Mordor. Corruption creeps up the longer
the Ring is carried and spikes whenever a Nazgul draws near. When corruption crosses the
threshold, the Eye turns.

**How to use it**
1. Run with `DRY_RUN = True` first. It just prints events so you can watch the theme work.
2. Create an Eventstream, add a **Custom endpoint** source, and copy its Event Hub
   compatible connection string plus the hub name into the config cell.
3. Set `DRY_RUN = False` and run again to stream live into the Eventstream.
4. Point your Activator at that Eventstream and set the rule (see the last cell).


In [10]:
# Only needed when DRY_RUN is False. Safe to run either way.
!pip install azure-eventhub --quiet

StatementMeta(, 775f8322-ab14-4450-b7b4-fff4d01fe926, 18, Finished, Available, Finished, False)

In [ ]:
import json, time, random, datetime as dt

# ----------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------
# DRY_RUN = True  -> print events only, no endpoint needed
# DRY_RUN = False -> send to the Eventstream custom endpoint below
DRY_RUN = False

# Paste these from your Eventstream custom endpoint (Sample connection strings). The connection string is under SAS Key and will be the primary connection string.
# Never commit real connection strings to source control.
EVENTHUB_CONN_STR = "Endpoint=sb://**********************.servicebus.windows.net/;SharedAccessKeyName=key_********-****-****-****-************;SharedAccessKey=********************************************;EntityPath-*************************"
EVENTHUB_NAME     = "************************"



# Simulation controls
NUM_EVENTS      = 60      # how many ticks to generate
SECONDS_PER_TICK = 2      # delay between events
CORRUPTION_ALERT = 70     # the level your Activator rule should watch for

# The road to Mordor, roughly in order
WAYPOINTS = [
    "The Shire", "Bree", "Weathertop", "Rivendell", "Hollin",
    "Moria", "Lothlorien", "The Great River", "Amon Hen",
    "Emyn Muil", "Dead Marshes", "Black Gate", "Cirith Ungol", "Mount Doom",
]


StatementMeta(, 775f8322-ab14-4450-b7b4-fff4d01fe926, 19, Finished, Available, Finished, False)

In [12]:
def ring_journey(num_events):
    """Yield one corruption reading per tick as the Ringbearer moves toward Mordor."""
    corruption = 8.0            # starts low and precious
    total = len(WAYPOINTS)
    for i in range(num_events):
        # Progress along the road (0..1), used to pick a location
        progress = i / max(num_events - 1, 1)
        location = WAYPOINTS[min(int(progress * (total - 1)), total - 1)]

        # A Nazgul appears now and then, more often the closer you get to Mordor
        nazgul_nearby = random.random() < (0.05 + 0.20 * progress)

        # Corruption drifts up slowly, jumps hard when a Nazgul is near
        drift = 0.8 + random.uniform(-0.3, 0.6)
        spike = random.uniform(12, 22) if nazgul_nearby else 0.0
        corruption = max(0.0, min(100.0, corruption + drift + spike))

        event = {
            "event_time": dt.datetime.utcnow().isoformat() + "Z",
            "ringbearer": "Frodo Baggins",
            "location": location,
            "corruption_level": round(corruption, 1),
            "nazgul_nearby": nazgul_nearby,
            "leagues_from_mordor": round(500 * (1 - progress), 1),
        }
        yield event


def render(event):
    """A little flavor for the printed output."""
    eye = "  <<< THE EYE TURNS" if event["corruption_level"] >= CORRUPTION_ALERT else ""
    nz = " (Nazgul near!)" if event["nazgul_nearby"] else ""
    return (f'{event["location"]:>16} | corruption {event["corruption_level"]:5.1f}'
            f' | {event["leagues_from_mordor"]:5.1f} leagues out{nz}{eye}')


StatementMeta(, 775f8322-ab14-4450-b7b4-fff4d01fe926, 20, Finished, Available, Finished, False)

In [13]:
# ----------------------------------------------------------------------
# STREAM
# ----------------------------------------------------------------------
if DRY_RUN:
    print("DRY RUN: printing events only. No data leaves the notebook.\n")
    for event in ring_journey(NUM_EVENTS):
        print(render(event))
        time.sleep(SECONDS_PER_TICK)
    print("\nThe quest is complete. Set DRY_RUN = False to stream for real.")
else:
    from azure.eventhub import EventHubProducerClient, EventData

    producer = EventHubProducerClient.from_connection_string(
        conn_str=EVENTHUB_CONN_STR,
        eventhub_name=EVENTHUB_NAME,
    )
    sent = 0
    try:
        with producer:
            for event in ring_journey(NUM_EVENTS):
                batch = producer.create_batch()
                batch.add(EventData(json.dumps(event)))
                producer.send_batch(batch)
                sent += 1
                print(render(event))
                time.sleep(SECONDS_PER_TICK)
    finally:
        print(f"\nSent {sent} events to the Eventstream. The Eye is watching.")


StatementMeta(, 775f8322-ab14-4450-b7b4-fff4d01fe926, 21, Finished, Available, Finished, False)

       The Shire | corruption   8.6 | 500.0 leagues out
       The Shire | corruption   9.6 | 491.5 leagues out
       The Shire | corruption  10.4 | 483.1 leagues out
       The Shire | corruption  11.2 | 474.6 leagues out
       The Shire | corruption  11.8 | 466.1 leagues out
            Bree | corruption  32.1 | 457.6 leagues out (Nazgul near!)
            Bree | corruption  32.7 | 449.2 leagues out
            Bree | corruption  33.8 | 440.7 leagues out
            Bree | corruption  34.8 | 432.2 leagues out
            Bree | corruption  36.0 | 423.7 leagues out
      Weathertop | corruption  37.2 | 415.3 leagues out
      Weathertop | corruption  38.6 | 406.8 leagues out
      Weathertop | corruption  39.5 | 398.3 leagues out
      Weathertop | corruption  40.8 | 389.8 leagues out
       Rivendell | corruption  41.7 | 381.4 leagues out
       Rivendell | corruption  42.8 | 372.9 leagues out
       Rivendell | corruption  43.6 | 364.4 leagues out
       Rivendell | corruption  44

## Wiring the Activator (in the Fabric UI)

The rule itself is set up in the Activator item, not in code. Once the events above are
flowing into your Eventstream:

1. Create an **Activator** item and add your Eventstream as the source.
2. Choose the event stream and assign an object. Use `ringbearer` as the object id so all
   readings for Frodo group together.
3. Add a property for `corruption_level`.
4. Create a rule: **when `corruption_level` becomes greater than 70**, then take an action
   (send a Teams message or email, or run a Fabric item).
5. Give the alert some flavor, for example:
   *"The Eye of Sauron has turned toward Middle-earth. The Ring has been detected at {location}."*

## The cost lesson hiding in this demo

This is the part that ties straight into your presentation. The moment that rule goes live,
Activator starts an **event listener** that runs continuously and consumes capacity by the
hour, whether or not any events arrive. Pausing or stopping the rule does **not** stop the
listener. Only deleting the rule stops the drain.

So this cheerful little demo rule is the One Ring. It looks small and precious, it quietly
consumes your capacity in the background, and it will keep doing so long after the quest is
over unless you destroy it. That is exactly the item we will hunt down and measure in the
next notebook.
